In [1]:
import numpy as np
import time
from pynq import Overlay, allocate, PL

# 1. Load the overlay
PL.reset()
ol = Overlay("loopback.bit")

In [2]:
 ol.ip_dict


{'axi_dma_0': {'type': 'xilinx.com:ip:axi_dma:7.1',
  'mem_id': 'S_AXI_LITE',
  'memtype': 'REGISTER',
  'gpio': {},
  'interrupts': {'mm2s_introut': {'controller': 'axi_intc_0',
    'index': 0,
    'fullpath': 'axi_dma_0/mm2s_introut'},
   's2mm_introut': {'controller': 'axi_intc_0',
    'index': 1,
    'fullpath': 'axi_dma_0/s2mm_introut'}},
  'parameters': {'C_S_AXI_LITE_ADDR_WIDTH': '10',
   'C_S_AXI_LITE_DATA_WIDTH': '32',
   'C_DLYTMR_RESOLUTION': '125',
   'C_PRMRY_IS_ACLK_ASYNC': '1',
   'C_ENABLE_MULTI_CHANNEL': '0',
   'C_NUM_MM2S_CHANNELS': '1',
   'C_NUM_S2MM_CHANNELS': '1',
   'C_INCLUDE_SG': '0',
   'C_SG_INCLUDE_STSCNTRL_STRM': '0',
   'C_SG_USE_STSAPP_LENGTH': '0',
   'C_SG_LENGTH_WIDTH': '26',
   'C_M_AXI_SG_ADDR_WIDTH': '64',
   'C_M_AXI_SG_DATA_WIDTH': '32',
   'C_M_AXIS_MM2S_CNTRL_TDATA_WIDTH': '32',
   'C_S_AXIS_S2MM_STS_TDATA_WIDTH': '32',
   'C_MICRO_DMA': '0',
   'C_INCLUDE_MM2S': '1',
   'C_INCLUDE_MM2S_SF': '1',
   'C_MM2S_BURST_SIZE': '128',
   'C_M_AXI_MM2S_

In [3]:
dma = ol.axi_dma_0
dma_send = ol.axi_dma_0.sendchannel

In [4]:
#without caliberation code


import xrfdc
import time
import numpy as np
from pynq import Overlay, allocate

# --- INITIALIZATION ---
# (Assumes Overlay 'ol' is already loaded)
rfdc = xrfdc.RFdc(ol.ip_dict['usp_rf_data_converter_0'])
target_block = rfdc.dac_tiles[0].blocks[0]
dma_send = ol.axi_dma_0.sendchannel
dma_recv = ol.axi_dma_0.recvchannel

# Constants
DATA_SIZE_RECV = 8192 
CARRIER_FREQ = 100 

# TARGET_RMS: These are the average "strengths" of your 0.5V, 1.0V, 1.5V, and 2.0V signals.
# You will need to calibrate these 4 numbers based on your first run.
# Update your constants with the real values you just measured
TARGET_RMS = [900, 1800, 2700, 3600] 
RMS_TO_BITS = {900: "00", 1800: "01", 2700: "10", 3600: "11"}

# Allocate Buffers
input_buffer = allocate(shape=(1024,), dtype=np.uint16)
output_buffer = allocate(shape=(DATA_SIZE_RECV,), dtype=np.int16)

def update_nco(rf_block, nco_freq):
    mixer_cfg = rf_block.MixerSettings
    mixer_cfg['Freq'] = nco_freq
    rf_block.MixerSettings = mixer_cfg
    rf_block.UpdateEvent(xrfdc.EVENT_MIXER)

def analyze_amplitude_time_domain(data):
    """Calculates the strength of the signal using RMS (Root Mean Square)"""
    # 1. Convert to float and remove DC offset
    float_data = data.astype(np.float32) - np.mean(data)
    
    # 2. Calculate RMS: sqrt(mean(squares))
    # This represents the average power/amplitude of the sine wave
    rms_val = np.sqrt(np.mean(float_data**2))
    return rms_val

def play_and_analyze_ask(sequence):
    print(f"Starting Time-Domain ASK Sequence: {sequence}")
    decoded_string = ""
    
    # 1. Set Carrier Frequency Once
    update_nco(target_block, CARRIER_FREQ)
    
    for i in range(0, len(sequence), 2):
        pair = sequence[i:i+2]
        
        # --- TRANSMIT ---
        if pair == "00": amp = 8192    # 0.25V target
        elif pair == "01": amp = 16384 # 0.5V target
        elif pair == "10": amp = 24576 # 0.75V target
        elif pair == "11": amp = 32767 # 1.0V target
        else: continue
            
        input_buffer[:] = amp
        dma_send.transfer(input_buffer)
        
        time.sleep(0.1) # Stabilization
        
        # --- RECEIVE & DECODE ---
        dma_recv.transfer(output_buffer)
        dma_recv.wait()
        
        detected_rms = analyze_amplitude_time_domain(output_buffer)
        
        # Find the closest matching RMS level
        closest_rms = min(TARGET_RMS, key=lambda x: abs(x - detected_rms))
        detected_bits = RMS_TO_BITS[closest_rms]
        
        decoded_string += detected_bits
        print(f"TX: {pair} -> RX RMS: {detected_rms:.1f} -> Decoded: {detected_bits}")
        
       # time.sleep(3)

    # 3. SHUTDOWN
    input_buffer[:] = 0 
    
    dma_send.transfer(input_buffer) 
    print(f"\nFinal Decoded String: {decoded_string}")

# --- Run ---
try:
    user_seq = input("Enter binary sequence: ")
    play_and_analyze_ask(user_seq)
except KeyboardInterrupt:
    print("\nStopped.")

Enter binary sequence:  00011011


Starting Time-Domain ASK Sequence: 00011011
TX: 00 -> RX RMS: 902.4 -> Decoded: 00
TX: 01 -> RX RMS: 1818.9 -> Decoded: 01
TX: 10 -> RX RMS: 2741.2 -> Decoded: 10
TX: 11 -> RX RMS: 3655.7 -> Decoded: 11

Final Decoded String: 00011011


In [24]:
#with caliberation part 1

import xrfdc
import time
import numpy as np
from pynq import Overlay, allocate

ol = Overlay("loopback.bit")
rfdc = xrfdc.RFdc(ol.ip_dict['usp_rf_data_converter_0'])
target_block = rfdc.dac_tiles[0].blocks[0]
dma_send = ol.axi_dma_0.sendchannel
dma_recv = ol.axi_dma_0.recvchannel

# Buffers
input_buffer = allocate(shape=(1024,), dtype=np.uint16)
output_buffer = allocate(shape=(8192,), dtype=np.int16)

def update_nco(rf_block, nco_freq):
    mixer_cfg = rf_block.MixerSettings
    mixer_cfg['Freq'] = nco_freq
    rf_block.MixerSettings = mixer_cfg
    rf_block.UpdateEvent(xrfdc.EVENT_MIXER)

def analyze_amplitude_time_domain(data):
    float_data = data.astype(np.float32) - np.mean(data)
    return np.sqrt(np.mean(float_data**2))

In [30]:
#with caliberation part 2

CARRIER_FREQ = 100 # Change this to test the limit (e.g., 100, 400, 600)

print(f"--- Calibrating Channel at {CARRIER_FREQ} MHz ---")
update_nco(target_block, CARRIER_FREQ)

# Send Pilot Signal (Full Scale 11)
input_buffer[:] = 32767
dma_send.transfer(input_buffer)
time.sleep(0.2) 

# Measure
dma_recv.transfer(output_buffer)
dma_recv.wait()
max_rms = analyze_amplitude_time_domain(output_buffer)

# Save the 'Ruler' globally so Cell 3 can see it
my_targets = [max_rms * 0.25, max_rms * 0.50, max_rms * 0.75, max_rms]
print(f"Calibration Saved. Max RMS at this freq: {max_rms:.1f}")

--- Calibrating Channel at 100 MHz ---
Calibration Saved. Max RMS at this freq: 3620.9


In [31]:
#with caliberation part 3

user_seq = input("Enter binary sequence: ")
decoded_string = ""
bit_map = ["00", "01", "10", "11"]

for i in range(0, len(user_seq), 2):
    pair = user_seq[i:i+2]
    amp = 8192 if pair=="00" else 16384 if pair=="01" else 24576 if pair=="10" else 32767
    
    input_buffer[:] = amp
    dma_send.transfer(input_buffer)
    #time.sleep(0.1)
    
    dma_recv.transfer(output_buffer)
    dma_recv.wait()
    
    current_rms = analyze_amplitude_time_domain(output_buffer)
    idx = np.argmin([abs(current_rms - t) for t in my_targets])
    
    decoded_string += bit_map[idx]
    print(f"TX {pair} -> RX {bit_map[idx]} (RMS: {current_rms:.1f})")

print(f"\nFinal Result: {decoded_string}")

Enter binary sequence:  10001010010101001001101101011


TX 10 -> RX 10 (RMS: 2791.6)
TX 00 -> RX 00 (RMS: 1024.4)
TX 10 -> RX 10 (RMS: 2721.5)
TX 10 -> RX 10 (RMS: 2760.2)
TX 01 -> RX 01 (RMS: 1874.1)
TX 01 -> RX 01 (RMS: 1839.9)
TX 01 -> RX 01 (RMS: 1839.8)
TX 00 -> RX 00 (RMS: 960.8)
TX 10 -> RX 10 (RMS: 2720.5)
TX 01 -> RX 01 (RMS: 1876.0)
TX 10 -> RX 10 (RMS: 2736.2)
TX 11 -> RX 11 (RMS: 3652.6)
TX 01 -> RX 01 (RMS: 1926.6)
TX 01 -> RX 01 (RMS: 1840.7)
TX 1 -> RX 11 (RMS: 3634.8)

Final Result: 100010100101010010011011010111


In [32]:
#bit rate calculation


import time

# 1. User Input
user_seq = input("Enter binary sequence: ")
num_bits = len(user_seq)

if num_bits % 2 != 0:
    print("Error: Please enter an even number of bits.")
else:
    # 2. Start Timing
    start_time = time.perf_counter()

    decoded_string = ""
    bit_map = ["00", "01", "10", "11"]

    print(f"--- Starting Transmission of {num_bits} bits ---")

    for i in range(0, num_bits, 2):
        pair = user_seq[i:i+2]
        
        # Select Amplitude
        if pair == "00": amp = 8192
        elif pair == "01": amp = 16384
        elif pair == "10": amp = 24576
        elif pair == "11": amp = 32767
        
        # DAC Send
        input_buffer[:] = amp
        dma_send.transfer(input_buffer)
        
        # ADC Receive
        dma_recv.transfer(output_buffer)
        dma_recv.wait()
        
        # Decode using RMS
        current_rms = analyze_amplitude_time_domain(output_buffer)
        
        # Logic to find the closest calibrated target
        idx = np.argmin([abs(current_rms - t) for t in my_targets])
        decoded_string += bit_map[idx]

    # 3. End Timing
    end_time = time.perf_counter()
    total_duration = end_time - start_time

    # 4. Calculate Accuracy
    # Compare original string to decoded string bit-by-bit
    correct_bits = sum(1 for a, b in zip(user_seq, decoded_string) if a == b)
    accuracy_pct = (correct_bits / num_bits) * 100

    # 5. Calculate Data Rate
    bps = num_bits / total_duration

    print("\n" + "="*45)
    print(f"RESULTS FOR CARRIER {CARRIER_FREQ} MHz")
    print("="*45)
    print(f"Original:         {user_seq}")
    print(f"Decoded:          {decoded_string}")
    print("-" * 45)
    print(f"Total Bits:       {num_bits}")
    print(f"Correct Bits:     {correct_bits}")
    print(f"Accuracy:         {accuracy_pct:.2f}%")
    print(f"Total Time:       {total_duration:.4f} s")
    print(f"Data Rate:        {bps:.2f} bps")
    print("="*45)

Enter binary sequence:  100010100101010010011011010110


--- Starting Transmission of 30 bits ---

RESULTS FOR CARRIER 100 MHz
Original:         100010100101010010011011010110
Decoded:          100010100101010010011011010110
---------------------------------------------
Total Bits:       30
Correct Bits:     30
Accuracy:         100.00%
Total Time:       0.0314 s
Data Rate:        953.98 bps


In [29]:
from pynq import Clocks

# Check individual PL clocks (values are in MHz)
print(f"PL Clock 0: {Clocks.fclk0_mhz} MHz")
print(f"PL Clock 1: {Clocks.fclk1_mhz} MHz")
print(f"PL Clock 2: {Clocks.fclk2_mhz} MHz")
print(f"PL Clock 3: {Clocks.fclk3_mhz} MHz")

# You can also check the CPU clock
print(f"CPU Clock: {Clocks.cpu_mhz} MHz")

PL Clock 0: 299.997 MHz
PL Clock 1: 99.999 MHz
PL Clock 2: 99.999 MHz
PL Clock 3: 99.999 MHz
CPU Clock: 1199.988 MHz


In [19]:
Clocks.fclk0_mhz = 300.0  # Sets PL Clock 0 to 150 MHz